# Import

In [175]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Đọc dữ liệu

In [176]:
train = pd.read_csv(
    "../data/train.csv",
    keep_default_na=False
)

test = pd.read_csv(
    "../data/test.csv",
    keep_default_na=False
)

print("Train:", train.shape)
print("Test :", test.shape)

Train: (1460, 81)
Test : (1459, 80)


# Phân loại biến

# Kiểm tra trùng lập dữ liệu

In [177]:
print("Train duplicate:", train.duplicated().sum())
print("Test duplicate :", test.duplicated().sum())

Train duplicate: 0
Test duplicate : 0


In [178]:
train = train.replace(r'^\s*$', np.nan, regex=True)
test = test.replace(r'^\s*$', np.nan, regex=True)

# Kiểm tra missing 

In [179]:
missing_train = train.isna().sum()
missing_test = test.isna().sum()

print("Missing trong train:")
print(missing_train[missing_train > 0])

print("\nMissing trong test:")
print(missing_test[missing_test > 0])

Missing trong train:
Series([], dtype: int64)

Missing trong test:
Series([], dtype: int64)


In [180]:
missing_ratio = (
    train.isna().mean() * 100
).sort_values(ascending=False)

print(missing_ratio[missing_ratio > 0])

Series([], dtype: float64)


# Tách thuộc tính dích khỏi feature

In [181]:
X = train.drop(columns=["Id", "SalePrice"])
y = train["SalePrice"]

X_test = test.drop(columns=["Id"])

print("X:", X.shape)
print("y:", y.shape)
print("X_test:", X_test.shape)

X: (1460, 79)
y: (1460,)
X_test: (1459, 79)


# Chia train và validation

In [182]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("y_train:", y_train.shape)
print("y_val  :", y_val.shape)

X_train: (1168, 79)
X_val  : (292, 79)
y_train: (1168,)
y_val  : (292,)


# Phân loại biến 

In [183]:
continuous_vars = [
    "LotFrontage", "LotArea", "MasVnrArea",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
    "TotalBsmtSF", "1stFlrSF", "2ndFlrSF",
    "LowQualFinSF", "GrLivArea", "GarageArea",
    "WoodDeckSF", "OpenPorchSF", "EnclosedPorch",
    "3SsnPorch", "ScreenPorch", "PoolArea", "MiscVal"
]

discrete_vars = [
    "BsmtFullBath", "BsmtHalfBath",
    "FullBath", "HalfBath",
    "BedroomAbvGr", "KitchenAbvGr",
    "TotRmsAbvGrd", "Fireplaces",
    "GarageCars", "GarageYrBlt",
    "YearBuilt", "YearRemodAdd",
    "MoSold", "YrSold"
]

nominal_vars = [
    "MSSubClass", "MSZoning", "Street", "Alley",
    "LandContour", "LotConfig", "Neighborhood",
    "Condition1", "Condition2", "BldgType",
    "HouseStyle", "RoofStyle", "RoofMatl",
    "Exterior1st", "Exterior2nd", "MasVnrType",
    "Foundation", "Heating", "CentralAir",
    "GarageType", "MiscFeature", "SaleType",
    "SaleCondition", "Electrical"
]

ordinal_vars = [
    "LotShape", "Utilities", "LandSlope",
    "OverallQual", "OverallCond",
    "ExterQual", "ExterCond",
    "BsmtQual", "BsmtCond", "BsmtExposure",
    "BsmtFinType1", "BsmtFinType2",
    "HeatingQC", "KitchenQual", "Functional",
    "FireplaceQu", "GarageFinish",
    "GarageQual", "GarageCond",
    "PavedDrive", "PoolQC", "Fence"
]

# Kiểm tra phân loại

In [184]:
all_vars = (
    continuous_vars
    + discrete_vars
    + nominal_vars
    + ordinal_vars
)

print("Tổng số feature đã phân loại:", len(all_vars))
print("Số feature trong X:", len(X.columns))

print("\nFeature bị trùng:")
print(
    pd.Series(all_vars)[
        pd.Series(all_vars).duplicated()
    ].tolist()
)

print("\nFeature chưa phân loại:")
print(
    [col for col in X.columns if col not in all_vars]
)

Tổng số feature đã phân loại: 79
Số feature trong X: 79

Feature bị trùng:
[]

Feature chưa phân loại:
[]


# Ordinal Mapping

In [185]:
ordinal_mappings = {

    "LotShape": {
        "IR3": 1,
        "IR2": 2,
        "IR1": 3,
        "Reg": 4
    },

    
    "Utilities": {
        "ELO": 1,
        "NoSeWa": 2,
        "NoSewr": 3,
        "AllPub": 4
    },

    
    "LandSlope": {
        "Sev": 1,
        "Mod": 2,
        "Gtl": 3
    },

    "OverallQual": {
        1: 1,
        2: 2,
        3: 3,
        4: 4,
        5: 5,
        6: 6,
        7: 7,
        8: 8,
        9: 9,
        10: 10
    },


    "OverallCond": {
        1: 1,
        2: 2,
        3: 3,
        4: 4,
        5: 5,
        6: 6,
        7: 7,
        8: 8,
        9: 9,
        10: 10
    },

    
    "ExterQual": {
        "Po": 1,
        "Fa": 2,
        "TA": 3,
        "Gd": 4,
        "Ex": 5
    },


    "ExterCond": {
        "Po": 1,
        "Fa": 2,
        "TA": 3,
        "Gd": 4,
        "Ex": 5
    },

    
    "BsmtQual": {
        "NA": 0,
        "Po": 1,
        "Fa": 2,
        "TA": 3,
        "Gd": 4,
        "Ex": 5
    },


    "BsmtCond": {
        "NA": 0,
        "Po": 1,
        "Fa": 2,
        "TA": 3,
        "Gd": 4,
        "Ex": 5
    },

    # Mức độ tiếp xúc của tầng hầm
    "BsmtExposure": {
        "NA": 0,
        "No": 1,
        "Mn": 2,
        "Av": 3,
        "Gd": 4
    },

    # Loại hoàn thiện tầng hầm 1
    "BsmtFinType1": {
        "NA": 0,
        "Unf": 1,
        "LwQ": 2,
        "Rec": 3,
        "BLQ": 4,
        "ALQ": 5,
        "GLQ": 6
    },

    # Loại hoàn thiện tầng hầm 2
    "BsmtFinType2": {
        "NA": 0,
        "Unf": 1,
        "LwQ": 2,
        "Rec": 3,
        "BLQ": 4,
        "ALQ": 5,
        "GLQ": 6
    },

    # Chất lượng hệ thống sưởi
    "HeatingQC": {
        "Po": 1,
        "Fa": 2,
        "TA": 3,
        "Gd": 4,
        "Ex": 5
    },

    # Chất lượng bếp
    "KitchenQual": {
        "Po": 1,
        "Fa": 2,
        "TA": 3,
        "Gd": 4,
        "Ex": 5
    },

    # Chức năng ngôi nhà
    "Functional": {
        "Sal": 1,
        "Sev": 2,
        "Maj2": 3,
        "Maj1": 4,
        "Mod": 5,
        "Min2": 6,
        "Min1": 7,
        "Typ": 8
    },

    # Chất lượng lò sưởi
    "FireplaceQu": {
        "NA": 0,
        "Po": 1,
        "Fa": 2,
        "TA": 3,
        "Gd": 4,
        "Ex": 5
    },

    # Mức độ hoàn thiện garage
    "GarageFinish": {
        "NA": 0,
        "Unf": 1,
        "RFn": 2,
        "Fin": 3
    },

    # Chất lượng garage
    "GarageQual": {
        "NA": 0,
        "Po": 1,
        "Fa": 2,
        "TA": 3,
        "Gd": 4,
        "Ex": 5
    },

    # Tình trạng garage
    "GarageCond": {
        "NA": 0,
        "Po": 1,
        "Fa": 2,
        "TA": 3,
        "Gd": 4,
        "Ex": 5
    },

    # Đường vào nhà
    "PavedDrive": {
        "N": 1,
        "P": 2,
        "Y": 3
    },

    # Chất lượng hồ bơi
    "PoolQC": {
        "NA": 0,
        "Fa": 1,
        "TA": 2,
        "Gd": 3,
        "Ex": 4
    },

    # Chất lượng hàng rào
    "Fence": {
        "NA": 0,
        "MnWw": 1,
        "GdWo": 2,
        "MnPrv": 3,
        "GdPrv": 4
    }
}

# Encode Ordinal 

In [186]:
def encode_ordinal(df):
    df = df.copy()

    for col, mapping in ordinal_mappings.items():
        df[col] = df[col].map(mapping)

    return df

In [187]:
X_train_ordinal = encode_ordinal(X_train[ordinal_vars])
X_val_ordinal = encode_ordinal(X_val[ordinal_vars])
X_test_ordinal = encode_ordinal(X_test[ordinal_vars])

print(X_train[ordinal_vars].head())


     LotShape Utilities LandSlope  OverallQual  OverallCond ExterQual  \
254       Reg    AllPub       Gtl            5            6        TA   
1066      IR1    AllPub       Gtl            6            7        Gd   
638       Reg    AllPub       Gtl            5            7        TA   
799       Reg    AllPub       Gtl            5            7        TA   
380       Reg    AllPub       Gtl            5            6        TA   

     ExterCond BsmtQual BsmtCond BsmtExposure  ... HeatingQC KitchenQual  \
254         Gd       TA       TA           No  ...        TA          TA   
1066        TA       Gd       TA           No  ...        Gd          TA   
638         TA       Fa       TA           No  ...        Gd          TA   
799         TA       Gd       TA           No  ...        Ex          Gd   
380         TA       TA       TA           No  ...        TA          Gd   

     Functional FireplaceQu GarageFinish GarageQual GarageCond PavedDrive  \
254         Typ          NA

In [188]:
print("NaN processed train:", X_train_processed.isna().sum().sum())
print("NaN processed val:", X_val_processed.isna().sum().sum())
print("NaN processed test:", X_test_processed.isna().sum().sum())

NaN processed train: 0
NaN processed val: 0
NaN processed test: 5


# Z - scorce các contiuous và discrete

In [189]:
numeric_vars = continuous_vars + discrete_vars

X_train[numeric_vars] = X_train[numeric_vars].apply(pd.to_numeric, errors="coerce")
X_val[numeric_vars] = X_val[numeric_vars].apply(pd.to_numeric, errors="coerce")
X_test[numeric_vars] = X_test[numeric_vars].apply(pd.to_numeric, errors="coerce")

numeric_imputer = SimpleImputer(strategy="median")

X_train_numeric = numeric_imputer.fit_transform(X_train[numeric_vars])
X_val_numeric = numeric_imputer.transform(X_val[numeric_vars])
X_test_numeric = numeric_imputer.transform( X_test[numeric_vars])

# Z-scoreA
scaler = StandardScaler()
X_train_numeric = scaler.fit_transform(X_train_numeric)
X_val_numeric = scaler.transform(X_val_numeric)
X_test_numeric = scaler.transform(X_test_numeric)

# One-hot nominal

In [190]:
nominal_imputer = SimpleImputer(strategy="most_frequent")

X_train_nominal = nominal_imputer.fit_transform(X_train[nominal_vars])
X_val_nominal = nominal_imputer.transform(X_val[nominal_vars])
X_test_nominal = nominal_imputer.transform(X_test[nominal_vars])

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False)

X_train_nominal = encoder.fit_transform(X_train_nominal)
X_val_nominal = encoder.transform(X_val_nominal)
X_test_nominal = encoder.transform(X_test_nominal)

# Nối tất cả feature 

In [191]:
X_train_processed = np.hstack([
    X_train_numeric,
    X_train_ordinal,
    X_train_nominal
])

X_val_processed = np.hstack([
    X_val_numeric,
    X_val_ordinal,
    X_val_nominal
])

X_test_processed = np.hstack([
    X_test_numeric,
    X_test_ordinal,
    X_test_nominal
])

# Tạo tên feature

In [192]:
nominal_feature_names = encoder.get_feature_names_out(
    nominal_vars
)

feature_names = np.concatenate([
    numeric_vars,
    ordinal_vars,
    nominal_feature_names
])

In [193]:
# Chuyển thành DataFrame

In [194]:
X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names
)

X_val_processed = pd.DataFrame(
    X_val_processed,
    columns=feature_names
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names
)

print("X_train:", X_train_processed.shape)
print("X_val:", X_val_processed.shape)
print("X_test:", X_test_processed.shape)

X_train: (1168, 239)
X_val: (292, 239)
X_test: (1459, 239)


# Lưu

In [195]:
processed_path = "../data/processed"

os.makedirs(
    processed_path,
    exist_ok=True
)

X_train_processed.to_csv(
    f"{processed_path}/X_train_processed.csv",
    index=False
)

X_val_processed.to_csv(
    f"{processed_path}/X_val_processed.csv",
    index=False
)

X_test_processed.to_csv(
    f"{processed_path}/X_test_processed.csv",
    index=False
)

y_train.reset_index(drop=True).to_csv(
    f"{processed_path}/y_train.csv",
    index=False
)

y_val.reset_index(drop=True).to_csv(
    f"{processed_path}/y_val.csv",
    index=False
)

print("Đã lưu dữ liệu processed.")

Đã lưu dữ liệu processed.
